# AI활용프로그래밍 Week 13 실습 노트북
## 여러 일을 동시에
### 기다리는 동안 다른 일 하기 · 제한 시간 게임 만들기

이 노트북은 **Google Colab**에서 실행합니다.
위에서부터 셀을 하나씩 `Shift + Enter` 로 실행하세요.

### 오늘 목표
1. 기다리는 동안 다른 일을 시킬 수 있다.
2. 쓰레드를 만들고 끝날 때까지 기다릴 수 있다.
3. 여럿이 같은 값을 고칠 때 생기는 문제를 안다.
4. 제한 시간이 있는 작은 게임을 만들 수 있다.

> ⚠️ **이번 주 주의** — 끝나지 않는 쓰레드는 칸 실행이 끝나도 계속 돕니다.
> 이상하면 칸 왼쪽 **■ 중지**, 그래도 안 되면 **[런타임] → [세션 다시 시작]**.

## 오늘의 구성

- 1) 기다리는 동안 멈춰 있는 문제
- 2) 쓰레드 만들고 기다리기
- 3) 같은 값을 함께 고치면 생기는 일
- 4) 미니게임 — 로딩 표시와 제한 시간
- 5) With AI — 위험한 곳 찾아내기

## 기다리는 동안 아무것도 못 합니다

- `sleep` 은 프로그램 **전체**를 멈춰 세웁니다.
- 그동안 화면도 안 바뀌고 입력도 못 받습니다.
- 게임이라면 그 시간 동안 얼어붙은 것과 같습니다.
- 그래서 '기다리는 일'을 따로 떼어 내야 합니다.

**실행 결과:**
```
시작
끝
```

In [ ]:
import time
print("시작")
time.sleep(1)
print("끝")

## 동시에 하려면 — 쓰레드

- 일하는 사람을 한 명 더 두는 것과 같습니다.
- 한 명은 기다리고 다른 한 명은 계속 일합니다.
- 파이썬에서는 이것을 **쓰레드**라고 부릅니다.
- 특히 기다림이 많은 일에 효과가 큽니다.

> 📖 **쓰레드(thread)** — 프로그램 안에서 따로 도는 일꾼 하나. 주방에서 한 명이 더 붙는 것과 같습니다.
> 📖 **동시성(concurrency)** — 여러 일을 번갈아 가며 진행하는 것. 요리하며 빨래를 돌리는 것과 같습니다.

## 쓰레드 만들어 실행하기

- `Thread` 에 시킬 일의 **이름**을 줍니다.
- 괄호를 붙이지 않습니다. `work()` 가 아니라 `work` 입니다.
- `start()` 가 일을 시작시킵니다.
- `join()` 이 끝날 때까지 기다립니다.

**실행 결과:**
```
작업 중
메인 끝
```

In [ ]:
import threading

def work():
    print("작업 중")

t = threading.Thread(target=work)
t.start()
t.join()
print("메인 끝")

## 기다리지 않으면 순서가 섞입니다

- 메인이 기다리지 않고 그냥 다음 줄로 갑니다.
- 그래서 `tick` 과 메인 문장이 뒤섞입니다.
- **여러 번 실행해** 순서가 바뀌는 것을 보세요.
- 결과가 매번 달라지니 정답을 적지 않았습니다.

In [ ]:
import threading, time

def bg():
    for i in range(3):
        print("tick")
        time.sleep(0.3)

t = threading.Thread(target=bg)
t.start()
print("메인은 계속 갑니다")
t.join()

## 코랩에서 쓰레드를 쓸 때

- 배경에서 찍는 `print` 가 **다음 칸 결과에 섞일 수** 있습니다.
- 끝나지 않는 쓰레드는 칸이 끝나도 계속 돕니다.
- 그래서 **끝내는 조건을 반드시** 넣어야 합니다.
- 이상하면 **[런타임] → [세션 다시 시작]** 을 씁니다.

## 배경으로 돌리기 (daemon)

- `daemon=True` 는 '메인이 끝나면 같이 끝'입니다.
- 이게 없으면 끝없이 도는 쓰레드가 남습니다.
- 몇 번 찍힐지는 매번 조금씩 다릅니다.
- 저장 같은 중요한 일은 `daemon` 으로 두지 않습니다.

In [ ]:
import threading, time

def bg():
    while True:
        print("tick")
        time.sleep(0.5)

t = threading.Thread(target=bg, daemon=True)
t.start()
time.sleep(2)
print("메인 끝")

## 여럿이 같은 값을 고치면

- 값을 하나 올리는 일은 사실 **세 걸음**입니다.
- ① 지금 값을 읽고 ② 1을 더하고 ③ 다시 씁니다.
- 두 일꾼이 ①을 동시에 하면 **같은 값을 읽습니다.**
- 그러면 두 번 더했는데 한 번만 오릅니다.

> 📖 **경쟁 상태(race condition)** — 누가 먼저 하느냐에 따라 답이 달라지는 상태. 두 사람이 같은 칸에 동시에 적는 것과 같습니다.

## 값이 깨지는 것을 눈으로 보기

- ②의 틈 동안 다른 일꾼이 같은 값을 읽습니다.
- 둘 다 0을 읽고 둘 다 1을 씁니다.
- 두 번 더했는데 1만 오른 셈입니다.
- **오류가 안 나고 답만 틀리니** 찾기 어렵습니다.

**실행 결과:** 일꾼 둘이 5번씩 더하면 `10` 이어야 하는데
```
5
```

In [ ]:
import threading, time

count = 0

def add():
    global count
    for _ in range(5):
        value = count      # ① 읽고
        time.sleep(0.01)   # ② 딴짓하고
        count = value + 1  # ③ 쓴다

ts = [threading.Thread(target=add) for _ in range(2)]
for t in ts:
    t.start()
for t in ts:
    t.join()
print(count)

## 자물쇠로 막기 (Lock)

- `with lock:` 안에는 **한 번에 한 명만** 들어갑니다.
- 먼저 들어간 일꾼이 나올 때까지 나머지는 기다립니다.
- ①②③이 끊기지 않고 한 묶음으로 끝납니다.
- 묶는 범위는 꼭 필요한 만큼만 **좁게** 잡습니다.

**실행 결과:**
```
10
```

In [ ]:
import threading, time

count = 0
lock = threading.Lock()

def add():
    global count
    for _ in range(5):
        with lock:
            value = count
            time.sleep(0.01)
            count = value + 1

ts = [threading.Thread(target=add) for _ in range(2)]
for t in ts:
    t.start()
for t in ts:
    t.join()
print(count)

## 자물쇠는 좁게 겁니다

- 자물쇠 안에 들어간 동안은 다른 일꾼이 멈춰 섭니다.
- 그래서 넓게 걸면 동시에 하는 의미가 사라집니다.
- 같이 고치는 값 근처만 감쌉니다.
- 가장 좋은 것은 **같이 고치는 값을 아예 줄이는 것**입니다.

> 📖 **임계구역(critical section)** — 한 번에 한 명만 들어가야 하는 코드 구간. 화장실 한 칸과 같습니다.

## 미니게임 1. 로딩 표시 만들기

1) 점을 찍는 일을 쓰레드로 돌립니다.
2) 메인은 그동안 다른 일을 합니다.
3) 메인이 끝나면 표시도 멈추게 합니다.
4) **끝내는 조건을 반드시** 넣습니다.

**성공 확인:** 점이 계속 찍히다가 `완료!` 가 나오며 멈추면 성공입니다.

In [ ]:
import threading, time

running = True

def spinner():
    while running:
        print(".", end="", flush=True)
        time.sleep(0.2)

t = threading.Thread(target=spinner)
t.start()

time.sleep(2)   # 메인 작업
running = False
t.join()
print("완료!")

## 미니게임 2. 제한 시간 퀴즈

1) 시간을 재는 쓰레드를 배경으로 돌립니다.
2) 메인은 답을 입력받습니다.
3) 입력이 끝난 뒤 시간 초과였는지 확인합니다.
4) 제한 시간을 바꿔 가며 해 봅니다.

**성공 확인:** 5초 안에 답하면 통과, 늦으면 시간 초과가 나옵니다.

> ⚠️ 이 칸은 **입력을 기다립니다.** 코랩 아래쪽 입력 칸에 답을 치고 Enter 를 누르세요.
> 타이머는 5초 뒤 `timeout` 을 True 로 바꿀 뿐, 입력 자체를 끊지는 **못합니다.**

In [ ]:
import threading, time

timeout = False

def timer():
    global timeout
    time.sleep(5)
    timeout = True

threading.Thread(target=timer, daemon=True).start()

ans = input("3 + 4 = ")

if timeout:
    print("시간 초과!")
else:
    print("입력:", ans)

## 더 알아보기 (시험 범위 아님)

- **Queue** — 값을 직접 고치지 않고 쪽지로 주고받기.
- **Event** — 끝내라는 신호를 깔끔하게 보내기.
- 둘 다 오늘 배운 것을 더 안전하게 만드는 도구입니다.
- 관심 있으면 찾아보되 **시험에는 나오지 않습니다.**
- 오늘은 `Thread` · `join` · `Lock` 세 가지면 충분합니다.

## With AI: 동시성은 검증이 특히 중요

- 쓰레드 코드는 **돌 때마다 결과가 달라질 수** 있습니다.
- 한 번 잘 돌았다고 맞는 코드가 아닙니다.
- AI에게 '어디가 위험한지' 먼저 물어봅니다.
- 받은 코드는 **여러 번 실행해** 확인합니다.
- **프롬프트 → 반영 → 검증**을 기록합니다.

> 📖 **프롬프트(prompt)** — AI에게 주는 질문이나 지시. 자세할수록 답이 좋아집니다.

## 프롬프트 템플릿 (위험한 곳 찾기)

```
① 제가 쓴 쓰레드 코드예요:
   (코드 전체를 붙여넣기)

② 하려던 일:
   5초 제한 퀴즈를 만들려고 해요.

③ 여러 쓰레드가 같이 건드리는 값이 있는지 알려 주세요.

④ 실행할 때마다 결과가 달라질 수 있는 곳을 짚어 주세요.

⑤ 끝내는 조건이 빠지지 않았는지 확인하고, 고칠 곳만 알려 주세요.
```

- ③ **같이 건드리는 값**이 곧 위험한 곳입니다.
- ④ 는 여러 번 실행해 **직접 확인**해야 합니다.
- ⑤ 끝내는 조건은 코랩에서 특히 중요합니다.

## 실습 1. 끝나지 않는 쓰레드 고치기

⚠️ **아래 칸은 실행하면 멈추지 않습니다.**
실행한 뒤 칸 왼쪽의 **■ 중지** 를 눌러 반드시 멈추세요.

1) 실행하면 `tick` 이 끝없이 나옵니다.
2) **■ 중지** 를 눌러 멈춥니다.
3) 무엇이 끝을 만들어 주지 않는지 찾습니다.
4) 끝내는 조건을 넣어 스스로 멈추게 합니다.

**성공 확인:** `tick` 이 몇 번 찍히고 스스로 멈추면 성공입니다.

In [ ]:
import threading, time

def bg():
    while True:
        print("tick")
        time.sleep(0.5)

t = threading.Thread(target=bg)
t.start()

In [ ]:
# 여기에 고쳐 쓰세요 (끝나는 조건 또는 daemon=True)

## 실습 2. 깨지는 값 고치기

1) 위의 '값이 깨지는 것을 눈으로 보기' 칸을 다시 실행해 `5` 를 확인합니다.
2) 왜 `10` 이 아닌지 **세 걸음**으로 설명해 봅니다.
3) 아래 칸에서 `with lock:` 을 알맞은 자리에 넣습니다.
4) **여러 번** 실행해 항상 `10` 이 나오는지 확인합니다.

**성공 확인:** 여러 번 실행해도 항상 `10` 이 나오면 성공입니다.

In [ ]:
import threading, time

count = 0
lock = threading.Lock()

def add():
    global count
    for _ in range(5):
        # TODO: 아래 세 줄을 with lock: 안으로 넣으세요
        value = count
        time.sleep(0.01)
        count = value + 1

ts = [threading.Thread(target=add) for _ in range(2)]
for t in ts:
    t.start()
for t in ts:
    t.join()
print(count)

## 실습 3. AI와 위험한 곳 점검하기

1) 미니게임 2의 코드를 AI에게 붙여 줍니다.
2) 위 템플릿으로 위험한 곳을 물어봅니다.
3) 받은 답 중 하나를 **직접 실험해** 확인합니다.
4) 확인한 것과 못 한 것을 나눠 적습니다.

**AI가 이렇게 말하면 직접 확인해 보세요**

| AI 문장 | 확인할 것 |
|---|---|
| "`timeout` 에 경쟁 상태가 있습니다" | 정말 그런가요? 쓰는 쪽은 하나뿐 아닌가요? |
| "타이머가 입력을 끊습니다" | 5초 뒤에 정말 입력이 끊기나요? 직접 해 보세요. |

⚠️ **그럴듯한 말과 확인된 사실은 다릅니다.**

✍️ **점검 결과** (아래에 직접 적으세요)

- AI가 짚은 위험 ①:  → 직접 확인한 결과:
- AI가 짚은 위험 ②:  → 직접 확인한 결과:
- 확인하지 못한 것:

## AI 답변 확인하는 법 (동시성편)

- 여러 번 실행해도 같은 결과가 나오는가?
- 끝내는 조건이 들어 있는가?
- 여러 쓰레드가 같이 고치는 값이 있는가?
- 자물쇠를 필요한 곳에만 좁게 걸었는가?
- 코드를 한 줄씩 내 말로 설명할 수 있는가?

## 미니 퀴즈 (5문항)

- 1. 쓰레드를 시작시키는 것은? (`start` / `join`)
- 2. 끝날 때까지 기다리는 것은?
- 3. 값 하나 올리는 일은 몇 걸음일까요?
- 4. 같이 고치는 값을 지키는 도구는?
- 5. `daemon=True` 는 무슨 뜻일까요?

✅ 정답을 여기에 적어 보세요:

- 1)
- 2)
- 3)
- 4)
- 5)

## 오늘 정리 & 다음 주

- 쓰레드는 기다림이 많은 일에 효과가 큽니다.
- `start` 로 시작하고 `join` 으로 기다립니다.
- 같이 고치는 값은 **자물쇠로 지킵니다.**
- 끝내는 조건이 없으면 코랩에서 계속 돕니다.
- 다음 주 — 내가 만든 프로그램 발표.

## 과제

- 필수 — **제한 시간 퀴즈**를 완성하기
  - 문제를 세 개 내고 맞힌 개수 세기
  - 제한 시간을 넘기면 오답으로 처리하기
  - **끝내는 조건을 반드시** 넣기
- **AI 기록(필수)**
  - 프롬프트: AI에게 무엇을 어떻게 물었는지
  - 반영: 받은 답에서 무엇을 고쳐 썼는지
  - 검증: 여러 번 실행해 무엇을 확인했는지
- 제출 ① 코랩 오른쪽 위 [공유] → 링크 권한을
  - '링크가 있는 모든 사용자(뷰어)'로 → 링크 제출
- 제출 ② [파일] → .ipynb 다운로드 → 학교 LMS(과제 제출 사이트)

### 과제 작성 공간

In [ ]:
# 제한 시간 퀴즈 (문제 3개, 맞힌 개수 세기)

**AI 기록**

- 프롬프트:
- 반영한 점:
- 검증한 방법(몇 번 실행했는지 포함):